# Minimal LoRA XLM-R Dual-Head Mixture

A deliberately simple multitask notebook:

- one `xlm-roberta-base` backbone
- one classifier head for `p(k | x)`
- one scalar regressor head for `s_r(x)`
- one gate `g(x)`
- classifier expectation `s_c(x) = sum_k k p(k | x)`
- mixed score `s(x) = g(x) s_r(x) + (1 - g(x)) s_c(x)`

Loss:

$$L = L_{CE} + \lambda L_{Huber}(s_r, y) + eta L_{Huber}(s, y) + \gamma(s_r - s_c)^2.$$

No custom `PreTrainedModel` subclass, no extra output keys, no fancy logging. Keep it boring until it learns.


In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import json
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Value
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_dual_head_mixture_xlmr"

SAMPLE_N = None  # use e.g. 2000 for a smoke test
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
WARMUP_STEPS = 100
MAX_GRAD_NORM = 1.0
FP16 = True

LAMBDA_REG = 0.25
BETA_MIX = 1.00
GAMMA_CONS = 0.05
HUBER_DELTA = 0.75

MAIN_LORA_R = 64
MAIN_LORA_ALPHA = 32
GATE_LORA_R = 8
GATE_LORA_ALPHA = 16

# Optional anti-collapse gate regularizer. Keep at 0 for the first run.
ETA_GATE = 0.0
GATE_TARGET_USAGE = 0.30

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
y_val = val_df["label"].to_numpy(dtype=int)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def tokenize(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["class_labels"] = [int(x) for x in batch["label"]]
    out["score_labels"] = [float(x) for x in batch["label"]]
    return out


def to_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("class_labels", Value("int64"))
    ds = ds.cast_column("score_labels", Value("float32"))
    ds.set_format("torch")
    return ds


train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)

## Model

In [ ]:
class DualHeadMixtureModel(nn.Module):
    def __init__(self, model_id=MODEL_ID, n_classes=N_CLASSES):
        super().__init__()
        self.n_classes = n_classes
        self.backbone = AutoModel.from_pretrained(model_id)
        main_lora_config = LoraConfig(
            r=MAIN_LORA_R,
            lora_alpha=MAIN_LORA_ALPHA,
            target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
            lora_dropout=0.01,
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.backbone = get_peft_model(self.backbone, main_lora_config)

        self.gate_backbone = AutoModel.from_pretrained(model_id)
        gate_lora_config = LoraConfig(
            r=GATE_LORA_R,
            lora_alpha=GATE_LORA_ALPHA,
            target_modules=["query", "key", "value"],
            lora_dropout=0.01,
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.gate_backbone = get_peft_model(self.gate_backbone, gate_lora_config)
        hidden = self.backbone.config.hidden_size
        head_hidden = hidden // 2
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, head_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(head_hidden, n_classes),
        )
        self.regressor = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, head_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(head_hidden, 1),
        )
        self.gate = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, head_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(head_hidden, 1),
        )
        self.register_buffer("class_values", torch.arange(n_classes, dtype=torch.float32), persistent=False)

        nn.init.normal_(self.classifier[-1].weight, std=0.02)
        nn.init.zeros_(self.classifier[-1].bias)
        nn.init.normal_(self.regressor[-1].weight, std=1e-3)
        nn.init.zeros_(self.regressor[-1].bias)  # 4 * sigmoid(0) = 2
        nn.init.normal_(self.gate[-1].weight, std=1e-3)
        gate_logit = np.log(GATE_TARGET_USAGE / (1.0 - GATE_TARGET_USAGE))
        nn.init.constant_(self.gate[-1].bias, float(gate_logit))

    def forward(self, input_ids=None, attention_mask=None, class_labels=None, score_labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        h = self.dropout(outputs.last_hidden_state[:, 0, :])
        gate_outputs = self.gate_backbone(input_ids=input_ids, attention_mask=attention_mask)
        h_gate = self.dropout(gate_outputs.last_hidden_state[:, 0, :])

        class_logits = self.classifier(h)
        probs = F.softmax(class_logits, dim=-1)
        s_c = probs @ self.class_values.to(probs.device)
        s_r = 4.0 * torch.sigmoid(self.regressor(h).squeeze(-1))
        gate = torch.sigmoid(self.gate(h_gate).squeeze(-1))
        s_mix = gate * s_r + (1.0 - gate) * s_c

        loss = None
        if class_labels is not None and score_labels is not None:
            y_class = class_labels.long().view(-1)
            y_score = score_labels.float().view(-1)
            loss_ce = F.cross_entropy(class_logits, y_class)
            loss_reg = F.huber_loss(s_r, y_score, delta=HUBER_DELTA)
            loss_mix = F.huber_loss(s_mix, y_score, delta=HUBER_DELTA)
            loss_cons = torch.square(s_r - s_c).mean()
            loss_gate = torch.square(gate.mean() - GATE_TARGET_USAGE)
            loss = loss_ce + LAMBDA_REG * loss_reg + BETA_MIX * loss_mix + GAMMA_CONS * loss_cons + ETA_GATE * loss_gate

        logits = torch.cat(
            [class_logits, s_r[:, None], s_c[:, None], s_mix[:, None], gate[:, None]],
            dim=-1,
        )
        return {"loss": loss, "logits": logits}


def make_model():
    model = DualHeadMixtureModel()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model

## Metrics and decoding

In [ ]:
def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(cls - classes), axis=1) for cls in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def apply_thresholds(scores, thresholds):
    return np.searchsorted(np.asarray(thresholds), np.asarray(scores), side="right").astype(int)


def tune_mae_thresholds(scores, labels, n_classes=N_CLASSES):
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_scores, starts = np.unique(sorted_scores, return_index=True)
    ends = np.r_[starts[1:], len(sorted_scores)]
    n_groups = len(unique_scores)

    group_cost = np.zeros((n_classes, n_groups), dtype=np.float64)
    for g, (start, end) in enumerate(zip(starts, ends)):
        y = sorted_labels[start:end]
        for cls in range(n_classes):
            group_cost[cls, g] = np.abs(cls - y).sum()

    prefix = np.c_[np.zeros(n_classes), np.cumsum(group_cost, axis=1)]
    dp = np.full((n_classes, n_groups + 1), np.inf)
    back = np.zeros((n_classes, n_groups + 1), dtype=int)
    dp[0] = prefix[0]
    for cls in range(1, n_classes):
        best_value = np.inf
        best_split = 0
        for j in range(n_groups + 1):
            candidate = dp[cls - 1, j] - prefix[cls, j]
            if candidate < best_value:
                best_value = candidate
                best_split = j
            dp[cls, j] = prefix[cls, j] + best_value
            back[cls, j] = best_split

    cuts = []
    j = n_groups
    for cls in range(n_classes - 1, 0, -1):
        j = back[cls, j]
        cuts.append(j)
    cuts = cuts[::-1]

    thresholds = []
    eps = 1e-6
    for cut in cuts:
        if cut <= 0:
            thresholds.append(float(unique_scores[0] - eps))
        elif cut >= n_groups:
            thresholds.append(float(unique_scores[-1] + eps))
        else:
            thresholds.append(float((unique_scores[cut - 1] + unique_scores[cut]) / 2.0))
    return np.array(thresholds, dtype=np.float32), apply_thresholds(scores, thresholds)


def split_predictions(predictions):
    pred = np.asarray(predictions, dtype=np.float64)
    class_logits = pred[:, :N_CLASSES]
    probs = softmax_np(class_logits)
    return {
        "class_logits": class_logits,
        "probs": probs,
        "s_r": np.clip(pred[:, N_CLASSES], 0, N_CLASSES - 1),
        "s_c": np.clip(pred[:, N_CLASSES + 1], 0, N_CLASSES - 1),
        "s_mix": np.clip(pred[:, N_CLASSES + 2], 0, N_CLASSES - 1),
        "gate": np.clip(pred[:, N_CLASSES + 3], 0, 1),
    }


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    _, score_labels = labels
    y_true = np.asarray(score_labels).reshape(-1)
    p = split_predictions(predictions)
    map_preds = p["probs"].argmax(axis=1).astype(int)
    bayes_preds = bayes_mae_decode(p["probs"])
    reg_preds = np.rint(p["s_r"]).astype(int)
    mix_preds = np.rint(p["s_mix"]).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, map_preds)),
        "map_mae": float(mean_absolute_error(y_true, map_preds)),
        "bayes_mae": float(mean_absolute_error(y_true, bayes_preds)),
        "reg_rounded_mae": float(mean_absolute_error(y_true, reg_preds)),
        "mixed_rounded_mae": float(mean_absolute_error(y_true, mix_preds)),
        "reg_score_mae": float(mean_absolute_error(y_true, p["s_r"])),
        "mixed_score_mae": float(mean_absolute_error(y_true, p["s_mix"])),
        "mean_gate": float(p["gate"].mean()),
        "gate_std": float(p["gate"].std()),
        "head_agreement_mse": float(np.square(p["s_r"] - p["s_c"]).mean()),
    }

## Train

In [ ]:
def make_training_args(run_name):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        max_grad_norm=MAX_GRAD_NORM,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        label_names=["class_labels", "score_labels"],
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)


def preflight(model, dataset):
    batch = dataset.select(range(min(4, len(dataset))))[:]
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items() if hasattr(v, "to")}
    model.train()
    model.zero_grad(set_to_none=True)
    out = model(**batch)
    print("preflight loss:", float(out["loss"].detach().cpu()))
    print("preflight logits finite:", bool(torch.isfinite(out["logits"]).all().detach().cpu()))
    print("preflight ranges:", {
        "s_r": [float(out["logits"][:, N_CLASSES].min().detach().cpu()), float(out["logits"][:, N_CLASSES].max().detach().cpu())],
        "s_c": [float(out["logits"][:, N_CLASSES + 1].min().detach().cpu()), float(out["logits"][:, N_CLASSES + 1].max().detach().cpu())],
        "s_mix": [float(out["logits"][:, N_CLASSES + 2].min().detach().cpu()), float(out["logits"][:, N_CLASSES + 2].max().detach().cpu())],
        "gate": [float(out["logits"][:, N_CLASSES + 3].min().detach().cpu()), float(out["logits"][:, N_CLASSES + 3].max().detach().cpu())],
    })
    out["loss"].backward()
    print("preflight grad finite:", all(p.grad is None or torch.isfinite(p.grad).all().item() for p in model.parameters()))
    model.zero_grad(set_to_none=True)


model = make_model()
if torch.cuda.is_available():
    model.to("cuda")
preflight(model, train_ds)

trainer = Trainer(
    model=model,
    args=make_training_args("dual_head_mixture_1epoch"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

## Validation diagnostics

In [ ]:
val_pred = trainer.predict(val_ds).predictions
p = split_predictions(val_pred)

map_labels = p["probs"].argmax(axis=1).astype(int)
bayes_labels = bayes_mae_decode(p["probs"])
reg_round = np.rint(p["s_r"]).astype(int)
mix_round = np.rint(p["s_mix"]).astype(int)
reg_thresholds, reg_tuned = tune_mae_thresholds(p["s_r"], y_val)
mix_thresholds, mix_tuned = tune_mae_thresholds(p["s_mix"], y_val)

summary = pd.DataFrame([
    {"decoder": "classifier_map", "mae": mean_absolute_error(y_val, map_labels), "counts": np.bincount(map_labels, minlength=N_CLASSES).tolist()},
    {"decoder": "classifier_bayes_mae", "mae": mean_absolute_error(y_val, bayes_labels), "counts": np.bincount(bayes_labels, minlength=N_CLASSES).tolist()},
    {"decoder": "regressor_round", "mae": mean_absolute_error(y_val, reg_round), "counts": np.bincount(reg_round, minlength=N_CLASSES).tolist()},
    {"decoder": "regressor_tuned", "mae": mean_absolute_error(y_val, reg_tuned), "counts": np.bincount(reg_tuned, minlength=N_CLASSES).tolist()},
    {"decoder": "mixture_round", "mae": mean_absolute_error(y_val, mix_round), "counts": np.bincount(mix_round, minlength=N_CLASSES).tolist()},
    {"decoder": "mixture_tuned", "mae": mean_absolute_error(y_val, mix_tuned), "counts": np.bincount(mix_tuned, minlength=N_CLASSES).tolist()},
])
display(summary.sort_values("mae"))

print("reg thresholds:", reg_thresholds.tolist())
print("mix thresholds:", mix_thresholds.tolist())
print("gate mean/std:", float(p["gate"].mean()), float(p["gate"].std()))
print("gate quantiles:", dict(zip([0, 0.1, 0.5, 0.9, 1.0], np.quantile(p["gate"], [0, 0.1, 0.5, 0.9, 1.0]).round(4))))
print("head agreement MSE:", float(np.square(p["s_r"] - p["s_c"]).mean()))

## Save

In [ ]:
final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), final_dir / "model.pt")
tokenizer.save_pretrained(str(final_dir))

config_path = OUTPUT_DIR / "dual_head_mixture_config.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps({
    "model_id": MODEL_ID,
    "n_classes": N_CLASSES,
    "lambda_reg": LAMBDA_REG,
    "beta_mix": BETA_MIX,
    "gamma_cons": GAMMA_CONS,
    "huber_delta": HUBER_DELTA,
    "main_lora_r": MAIN_LORA_R,
    "main_lora_alpha": MAIN_LORA_ALPHA,
    "gate_lora_r": GATE_LORA_R,
    "gate_lora_alpha": GATE_LORA_ALPHA,
    "eta_gate": ETA_GATE,
    "gate_target_usage": GATE_TARGET_USAGE,
    "reg_thresholds": reg_thresholds.tolist() if "reg_thresholds" in globals() else None,
    "mix_thresholds": mix_thresholds.tolist() if "mix_thresholds" in globals() else None,
    "recommended_decoder": "mixture_tuned",
}, indent=2), encoding="utf-8")
print("model:", final_dir / "model.pt")
print("config:", config_path)

## Optional submission

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_pred = trainer.predict(test_ds).predictions
    test_p = split_predictions(test_pred)
    test_bayes = bayes_mae_decode(test_p["probs"])
    test_mix = apply_thresholds(test_p["s_mix"], mix_thresholds if "mix_thresholds" in globals() else np.arange(0.5, N_CLASSES - 1, 1.0))

    bayes_path = OUTPUT_DIR / "submission_dual_head_bayes_mae.csv"
    mix_path = OUTPUT_DIR / "submission_dual_head_mixture_tuned.csv"
    pd.DataFrame({"id": test_df["id"], "label": test_bayes.astype(int)}).to_csv(bayes_path, index=False)
    pd.DataFrame({"id": test_df["id"], "label": test_mix.astype(int)}).to_csv(mix_path, index=False)
    print("bayes counts:", np.bincount(test_bayes, minlength=N_CLASSES).tolist())
    print("mix counts:", np.bincount(test_mix, minlength=N_CLASSES).tolist())
    print("gate mean/std:", float(test_p["gate"].mean()), float(test_p["gate"].std()))
    print(bayes_path)
    print(mix_path)
else:
    print("No test CSV found:", TEST_CSV)